# GA4 Churn Analysis — Google Colab

Bu notebook GA4 BigQuery export verisi üzerinde basit satın alma bazlı churn analizi çalıştırır.

**Akış:** paket kurulumu → kaynak bilgileri → Google Cloud yetkilendirmesi → maliyet kontrolü → temel tablo → satın alma aralıkları → churn analizi → HTML dashboard.

> BigQuery sorguları Colab belleğinde çalışmaz. Colab yalnızca sorguları BigQuery'ye gönderir; büyük veri BigQuery tarafında işlenir.


## 1. Paketi kur


In [ ]:
%pip install -q --upgrade "ga4-churn-toolkit @ git+https://github.com/yasinsariyildizz/ga4-churn-toolkit.git@main"


## 2. Kaynak bilgilerini gir

Aşağıdaki dört alanı kendi GA4 BigQuery export yapına göre doldur.


In [ ]:
PROJECT_ID = "your-gcp-project" # @param {type:"string"}
DATASET_ID = "analytics_123456789" # @param {type:"string"}
TABLE_ID = "events_*" # @param {type:"string"}
OUTPUT_DATASET_ID = "ga4_churn" # @param {type:"string"}


## 3. Google Cloud hesabınla yetkilendir

Bu hücre çalışınca Google hesabını seçmen istenir. Seçtiğin hesabın kaynak GA4 dataset'ini okuyabilmesi, BigQuery sorgusu çalıştırabilmesi ve çıktı dataset'ine tablo yazabilmesi gerekir.


In [ ]:
from google.colab import auth

auth.authenticate_user(project_id=PROJECT_ID)
print("Google Cloud yetkilendirmesi tamamlandı.")


## 4. Analizi başlat


In [ ]:
from ga4_churn import ChurnAnalysis

analysis = ChurnAnalysis(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    table_id=TABLE_ID,
    output_dataset_id=OUTPUT_DATASET_ID,
)

print("Kaynak:", analysis.source)
print("Çıktı dataset:", analysis.out)


## 5. Maliyet kontrolü

Henüz analiz tablosu oluşturmaz. BigQuery'nin yaklaşık ne kadar veri tarayacağını gösterir.


In [ ]:
analysis.dry_run()


## 6. Kullanıcı bazlı temel tabloyu oluştur

Her `user_pseudo_id` için alışveriş, oturum, gelir ve son alışverişten bu yana geçen gün bilgilerini hazırlar.


In [ ]:
analysis.create_base_table()


## 7. Satın alma aralıklarını incele

Tekrar alışveriş yapan kullanıcıların iki alışveriş günü arasında kaç gün geçtiğini analiz eder. Churn için kullanacağın gün sayısını seçmeden önce bu çıktıları incele.


In [ ]:
analysis.purchase_day_distribution()


## 8. Churn analizini çalıştır

Churn için kullanmak istediğin gün sayısını burada seç. Örneğin `90`, son alışverişinin üzerinden 90 günden fazla geçen alışveriş yapmış kullanıcıların churn olmuş sayılması demektir.


In [ ]:
CHURN_THRESHOLD = 90 # @param {type:"integer"}

churn_result = analysis.churn_analysis(CHURN_THRESHOLD)


## 9. HTML dashboard'u Colab içinde görüntüle

Bu hücre churn analizi sonunda oluşturulan HTML raporu notebook içinde açar.


In [ ]:
from pathlib import Path
from IPython.display import HTML, display

dashboard_path = Path(churn_result["dashboard_path"])
display(HTML(dashboard_path.read_text(encoding="utf-8")))


## 10. HTML dashboard'u indir

Dosyayı bilgisayarına indirmek için bu hücreyi çalıştır.


In [ ]:
from google.colab import files

files.download(churn_result["dashboard_path"])


## Gerekli BigQuery yetkileri

Colab'da giriş yaptığın Google hesabının en az şu işlemleri yapabilmesi gerekir:

- kaynak GA4 dataset'ini ve `events_*` tablolarını okuyabilmek,
- BigQuery sorgusu çalıştırabilmek,
- seçtiğin çıktı dataset'ini oluşturabilmek veya mevcut dataset'e tablo yazabilmek.

Yetki hatası alırsan notebook kodundan önce Google Cloud IAM izinlerini kontrol et.
